Grupitöö roll C RFM analüüsi osa

In [ ]:
# 1. Viitekuupäev
today = pd.to_datetime('2025-02-28')

# 2. Recency — viimase ostu kuupäev ja päevade arv tänaseni
recency_df = df.groupby('customer_id')['sale_date'].max().reset_index()
recency_df['recency_days'] = (today - recency_df['sale_date']).dt.days

# 3. Frequency — ostude arv
frequency_df = df.groupby('customer_id')['sale_id'].count().reset_index()
frequency_df.columns = ['customer_id', 'frequency']

# 4. Monetary — kogukulutus
monetary_df = df.groupby('customer_id')['total_price'].sum().reset_index()
monetary_df.columns = ['customer_id', 'monetary_value']

# 5. Liidame R, F, M ühte rfm tabelisse (pd.merge)
rfm = pd.merge(
    recency_df[['customer_id', 'recency_days']], frequency_df, on='customer_id'
)
rfm = pd.merge(rfm, monetary_df, on='customer_id')

# 6. Määrame skoorid 1-5 (pd.qcut)
# Recency: vastupidine (vähem päevi = skoor 5)
rfm['R_score'] = pd.qcut(
    rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1]
).astype(int)

# Frequency: vajab rank(method='first') duplikaatide tõttu
rfm['F_score'] = (
    pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
    .astype(int)
)

# Monetary: suurem summa = skoor 5
rfm['M_score'] = pd.qcut(
    rfm['monetary_value'], 5, labels=[1, 2, 3, 4, 5]
).astype(int)

# 7. Arvutame koguskoori ja määrame segmendid
rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

def get_segment(score):
  if score >= 13:
    return 'VIP Champions'
  elif score >= 10:
    return 'Loyal'
  elif score >= 7:
    return 'Potential'
  elif score >= 4:
    return 'At Risk'
  else:
    return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(get_segment)

# 8. Trükime segmentide kokkuvõtte
summary = pd.DataFrame({
    'Arv': rfm['Segment'].value_counts(),
    'Osakaal (%)': (rfm['Segment'].value_counts(normalize=True) * 100).round(1),
})

print('=== RFM SEGMENTIDE KOKKUVÕTE ===')
print(summary)